In [1]:
import os

base_dir = "/kaggle/input/datasets/srabondcosta/aqua-plant-final-dataset/AquPlantDS_FINAL"

print(os.listdir(base_dir))

['Validation', 'Test', 'Train']


In [2]:
train_dir = os.path.join(base_dir, "Train")
val_dir = os.path.join(base_dir, "Validation")
test_dir = os.path.join(base_dir, "Test")

print("Train exists:", os.path.exists(train_dir))
print("Validation exists:", os.path.exists(val_dir))
print("Test exists:", os.path.exists(test_dir))

Train exists: True
Validation exists: True
Test exists: True


In [3]:
import os
import random
import numpy as np
import torch

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# ============================================
# 1. REPRODUCIBILITY
# ============================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================
# 2. DEVICE
# ============================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================
# 3. PATHS
# ============================================

base_dir = "/kaggle/input/datasets/srabondcosta/aqua-plant-final-dataset/AquPlantDS_FINAL"

train_dir = os.path.join(base_dir, "Train")
val_dir = os.path.join(base_dir, "Validation")
test_dir = os.path.join(base_dir, "Test")


# ============================================
# 4. SETTINGS
# ============================================

IMG_SIZE = 224
BATCH_SIZE = 32

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]


# ============================================
# 5. TRAIN TRANSFORM
# ============================================

train_transform = transforms.Compose([

    transforms.RandomRotation(
        degrees=10,
        fill=(124, 116, 104)
    ),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.85, 1.0)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])


# ============================================
# 6. VALIDATION / TEST TRANSFORM
# ============================================

eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])


# ============================================
# 7. DATASETS
# ============================================

train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    val_dir,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=eval_transform
)


# ============================================
# 8. DATALOADERS
# ============================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================
# 9. VERIFICATION
# ============================================

print("\n==============================")
print("KAGGLE DATA PIPELINE")
print("==============================")

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("Classes:", len(train_dataset.classes))

images, labels = next(iter(train_loader))

print("\nBatch shape:", images.shape)
print("Label shape:", labels.shape)

print("\nDevice:", device)

print("\nDATA PIPELINE READY ✅")


KAGGLE DATA PIPELINE
Train: 744
Validation: 161
Test: 157
Classes: 14

Batch shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32])

Device: cuda

DATA PIPELINE READY ✅


In [4]:
import time
import copy
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

NUM_CLASSES = 14
EPOCHS = 100
LEARNING_RATE = 0.001
PATIENCE = 10

print("Device:", device)
print("Classes:", NUM_CLASSES)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

Device: cuda
Classes: 14
Epochs: 100
Learning rate: 0.001


In [5]:
class CustomCNN(nn.Module):

    def __init__(self, num_classes=14):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CustomCNN(NUM_CLASSES).to(device)

print(model)


CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("Loss: CrossEntropyLoss")
print("Optimizer: AdamW")
print("Scheduler: ReduceLROnPlateau")

Loss: CrossEntropyLoss
Optimizer: AdamW
Scheduler: ReduceLROnPlateau


In [7]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)

    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


print("Training function ready ✅")

Training function ready ✅


In [8]:
def validate_one_epoch(model, val_loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predicted.cpu().numpy()
            )

    val_loss = running_loss / len(val_loader.dataset)

    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    val_precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    val_recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return (
        val_loss,
        val_accuracy,
        val_precision,
        val_recall,
        val_f1
    )


print("Validation function ready ✅")

Validation function ready ✅


In [9]:
# ============================================
# RESET CUSTOM CNN FOR FINAL BASELINE RUN
# ============================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import time
import pandas as pd
import os

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Fresh Custom CNN
model = CustomCNN(num_classes=14).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

EPOCHS = 100
PATIENCE = 10

print("Fresh Custom CNN initialized ✅")
print("Device:", device)
print("Maximum epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Best-model criterion: Validation Macro-F1")

Fresh Custom CNN initialized ✅
Device: cuda
Maximum epochs: 100
Early stopping patience: 10
Best-model criterion: Validation Macro-F1


In [10]:
best_val_f1 = -1.0
best_epoch = 0
best_model_weights = None

epochs_without_improvement = 0
history = []

start_time = time.time()

best_model_path = "/kaggle/working/custom_cnn_best_f1.pth"
history_path = "/kaggle/working/custom_cnn_history.csv"


for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # =========================
    # VALIDATION
    # =========================
    (
        val_loss,
        val_accuracy,
        val_precision,
        val_recall,
        val_f1
    ) = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    # =========================
    # LR SCHEDULER
    # =========================
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # =========================
    # SAVE HISTORY
    # =========================
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_precision_macro": val_precision,
        "val_recall_macro": val_recall,
        "val_f1_macro": val_f1,
        "learning_rate": current_lr
    })

    # Save history every epoch
    pd.DataFrame(history).to_csv(
        history_path,
        index=False
    )

    # =========================
    # BEST MODEL BY MACRO-F1
    # =========================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        best_model_weights = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            best_model_weights,
            best_model_path
        )

        epochs_without_improvement = 0
        mark = " ⭐ BEST"

    else:

        epochs_without_improvement += 1
        mark = ""

    # =========================
    # PRINT RESULT
    # =========================
    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_accuracy:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_accuracy:.4f}"
        f" | Val F1: {val_f1:.4f}"
        f" | LR: {current_lr:.6f}"
        f"{mark}"
    )

    # =========================
    # EARLY STOPPING
    # =========================
    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping triggered: "
            f"no Macro-F1 improvement for {PATIENCE} epochs."
        )

        break


elapsed_time = time.time() - start_time

# Restore best Macro-F1 model
model.load_state_dict(best_model_weights)

print("\n====================================")
print("CUSTOM CNN TRAINING COMPLETE")
print("====================================")

print("Best Epoch:", best_epoch)

print(
    "Best Validation Macro-F1:",
    round(best_val_f1 * 100, 2),
    "%"
)

print(
    "Training time:",
    round(elapsed_time / 60, 2),
    "minutes"
)

print("\nBest model saved:")
print(best_model_path)

print("\nHistory saved:")
print(history_path)

print("\nTEST SET WAS NOT USED ✅")

Epoch 001/100 | Train Loss: 2.3049 | Train Acc: 0.2554 | Val Loss: 2.1539 | Val Acc: 0.3043 | Val F1: 0.2290 | LR: 0.001000 ⭐ BEST
Epoch 002/100 | Train Loss: 1.8388 | Train Acc: 0.4435 | Val Loss: 1.6101 | Val Acc: 0.4596 | Val F1: 0.4352 | LR: 0.001000 ⭐ BEST
Epoch 003/100 | Train Loss: 1.4809 | Train Acc: 0.5188 | Val Loss: 1.1765 | Val Acc: 0.6335 | Val F1: 0.6229 | LR: 0.001000 ⭐ BEST
Epoch 004/100 | Train Loss: 1.2364 | Train Acc: 0.5995 | Val Loss: 1.1236 | Val Acc: 0.6149 | Val F1: 0.6026 | LR: 0.001000
Epoch 005/100 | Train Loss: 1.1422 | Train Acc: 0.6210 | Val Loss: 0.9575 | Val Acc: 0.6957 | Val F1: 0.6817 | LR: 0.001000 ⭐ BEST
Epoch 006/100 | Train Loss: 1.0473 | Train Acc: 0.6532 | Val Loss: 1.4902 | Val Acc: 0.4907 | Val F1: 0.4561 | LR: 0.001000
Epoch 007/100 | Train Loss: 0.9013 | Train Acc: 0.7177 | Val Loss: 0.9707 | Val Acc: 0.6894 | Val F1: 0.6480 | LR: 0.001000
Epoch 008/100 | Train Loss: 0.9015 | Train Acc: 0.7056 | Val Loss: 0.6418 | Val Acc: 0.7702 | Val F1: 0.

In [11]:
import os
import pandas as pd

# ============================================
# FILE PATHS
# ============================================

history_path = "/kaggle/working/custom_cnn_history.csv"
model_path = "/kaggle/working/custom_cnn_best_f1.pth"


# ============================================
# CHECK SAVED FILES
# ============================================

print("====================================")
print("CUSTOM CNN FILE CHECK")
print("====================================")

print("History CSV exists:",
      os.path.exists(history_path))

print("Best model exists:",
      os.path.exists(model_path))


# ============================================
# LOAD TRAINING HISTORY
# ============================================

history_df = pd.read_csv(history_path)


# ============================================
# FIND BEST MACRO-F1 EPOCH
# ============================================

best_index = history_df["val_f1_macro"].idxmax()

best_row = history_df.loc[best_index]


# ============================================
# PRINT BEST RESULT
# ============================================

print("\n====================================")
print("CUSTOM CNN BEST VALIDATION RESULT")
print("====================================")

print(
    "Best Epoch:",
    int(best_row["epoch"])
)

print(
    "Validation Accuracy:",
    round(
        best_row["val_accuracy"] * 100,
        2
    ),
    "%"
)

print(
    "Validation Precision:",
    round(
        best_row["val_precision_macro"] * 100,
        2
    ),
    "%"
)

print(
    "Validation Recall:",
    round(
        best_row["val_recall_macro"] * 100,
        2
    ),
    "%"
)

print(
    "Validation Macro-F1:",
    round(
        best_row["val_f1_macro"] * 100,
        2
    ),
    "%"
)

print(
    "Validation Loss:",
    round(
        best_row["val_loss"],
        4
    )
)

print(
    "Train Accuracy:",
    round(
        best_row["train_accuracy"] * 100,
        2
    ),
    "%"
)

print(
    "Train Loss:",
    round(
        best_row["train_loss"],
        4
    )
)

print(
    "Learning Rate:",
    best_row["learning_rate"]
)


# ============================================
# MODEL FILE SIZE
# ============================================

if os.path.exists(model_path):

    model_size = os.path.getsize(
        model_path
    ) / (1024 * 1024)

    print(
        "Saved Model Size:",
        round(model_size, 2),
        "MB"
    )


print("\n====================================")
print("CUSTOM CNN BASELINE COMPLETE ✅")
print("Best model selected by Validation Macro-F1")
print("Test set was NOT used ✅")
print("====================================")

CUSTOM CNN FILE CHECK
History CSV exists: True
Best model exists: True

CUSTOM CNN BEST VALIDATION RESULT
Best Epoch: 22
Validation Accuracy: 96.89 %
Validation Precision: 97.14 %
Validation Recall: 96.92 %
Validation Macro-F1: 96.89 %
Validation Loss: 0.2362
Train Accuracy: 88.44 %
Train Loss: 0.3659
Learning Rate: 0.0005
Saved Model Size: 1.63 MB

CUSTOM CNN BASELINE COMPLETE ✅
Best model selected by Validation Macro-F1
Test set was NOT used ✅


In [12]:
import zipfile
import os

model_path = "/kaggle/working/custom_cnn_best_f1.pth"
history_path = "/kaggle/working/custom_cnn_history.csv"
zip_path = "/kaggle/working/Custom_CNN_Final_Artifacts.zip"

with zipfile.ZipFile(zip_path, "w") as z:
    z.write(
        model_path,
        arcname="custom_cnn_best_f1.pth"
    )
    z.write(
        history_path,
        arcname="custom_cnn_history.csv"
    )

print("ZIP created successfully ✅")
print(zip_path)
print(
    "ZIP size:",
    round(os.path.getsize(zip_path) / (1024 * 1024), 2),
    "MB"
)

ZIP created successfully ✅
/kaggle/working/Custom_CNN_Final_Artifacts.zip
ZIP size: 1.64 MB
